# Experiment: Dropout

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader

from torchvision import datasets
from torchvision.transforms import ToTensor

import matplotlib.pyplot as plt
import pandas as pd

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

torch.manual_seed(42)

In [2]:
train_data = datasets.FashionMNIST(
    root="../data",
    train=True,
    download=True,
    transform=ToTensor(),
    target_transform=None
)

test_data = datasets.FashionMNIST(
    root="../data",
    train=False,
    download=True,
    transform=ToTensor(),
    target_transform=None
)

print(f"Length of train data: {len(train_data)}")
print(f"Length of test data: {len(test_data)}")

Length of train data: 60000
Length of test data: 10000


In [3]:
class_names = train_data.classes
print(class_names)

['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


In [4]:
BATCH_SIZE = 32

train_dataloader = DataLoader(
    dataset=train_data,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_dataloader = DataLoader(
    dataset=test_data,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(f"Length of train_dataloader: {len(train_dataloader)}, No. of images per batch: {BATCH_SIZE}")
print(f"Length of test_dataloader: {len(test_dataloader)}, No. of images per batch: {BATCH_SIZE}")

Length of train_dataloader: 1875, No. of images per batch: 32
Length of test_dataloader: 313, No. of images per batch: 32


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")

Device : cuda


In [6]:
from torchmetrics import Accuracy, Precision, Recall, F1Score

train_metrics = {
    "accuracy": Accuracy(task="multiclass", num_classes=10).to(device),
    "precision": Precision(task="multiclass", num_classes=10, average="weighted").to(device),
    "recall": Recall(task="multiclass", num_classes=10, average="weighted").to(device),
    "f1": F1Score(task="multiclass", num_classes=10, average="weighted").to(device)
}

test_metrics = {
    "accuracy": Accuracy(task="multiclass", num_classes=10).to(device),
    "precision": Precision(task="multiclass", num_classes=10, average="weighted").to(device),
    "recall": Recall(task="multiclass", num_classes=10, average="weighted").to(device),
    "f1": F1Score(task="multiclass", num_classes=10, average="weighted").to(device)
}

exp_results = []

In [18]:
from src.models import CNN
from src.train import train_step, test_step

epochs = 20
history = {
    "train": [],
    "test": []
}

exp0 = CNN(dropout=0.0).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    params=exp0.parameters(),
    lr=0.0005
)

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    print("-"*30)

    train_results = train_step(
        model=exp0,
        dataloader=train_dataloader,
        loss_fn=loss_fn,
        optimizer=optimizer,
        metrics=train_metrics,
        device=device
    )

    test_results = test_step(
        model=exp0,
        dataloader=test_dataloader,
        loss_fn=loss_fn,
        metrics=test_metrics,
        device=device
    )

    history["train"].append(train_results)
    history["test"].append(test_results)

    print(
        f"Train Loss: {train_results['loss']:.4f} | "
        f"Train Acc: {train_results['accuracy']:.4f} | "
        f"Train Precision: {train_results['precision']:.4f} | "
        f"Train Recall: {train_results['recall']:.4f} | "
        f"Train F1: {train_results['f1']:.4f}"
    )

    print(
        f"Test Loss: {test_results['loss']:.4f} | "
        f"Test Acc: {test_results['accuracy']:.4f} | "
        f"Test Precision: {test_results['precision']:.4f} | "
        f"Test Recall: {test_results['recall']:.4f} | "
        f"Test F1: {test_results['f1']:.4f}"
    )

exp_results.append({
    "dropout": 0.0,
    "train_loss": history["train"][-1]["loss"],
    "test_loss": history["test"][-1]["loss"],
    "train_accuracy": history["train"][-1]["accuracy"],
    "test_accuracy": history["test"][-1]["accuracy"],
    "train_precision": history["train"][-1]["precision"],
    "test_precision": history["test"][-1]["precision"],
    "train_recall": history["train"][-1]["recall"],
    "test_recall": history["test"][-1]["recall"],
    "train_f1": history["train"][-1]["f1"],
    "test_f1": history["test"][-1]["f1"]
})


Epoch 1/20
------------------------------
Looked at 0 / 60000 samples
Looked at 12800 / 60000 samples
Looked at 25600 / 60000 samples
Looked at 38400 / 60000 samples
Looked at 51200 / 60000 samples
Train Loss: 0.4053 | Train Acc: 0.8556 | Train Precision: 0.8542 | Train Recall: 0.8556 | Train F1: 0.8546
Test Loss: 0.3224 | Test Acc: 0.8832 | Test Precision: 0.8882 | Test Recall: 0.8832 | Test F1: 0.8785

Epoch 2/20
------------------------------
Looked at 0 / 60000 samples
Looked at 12800 / 60000 samples
Looked at 25600 / 60000 samples
Looked at 38400 / 60000 samples
Looked at 51200 / 60000 samples
Train Loss: 0.2534 | Train Acc: 0.9084 | Train Precision: 0.9080 | Train Recall: 0.9084 | Train F1: 0.9081
Test Loss: 0.2671 | Test Acc: 0.9037 | Test Precision: 0.9062 | Test Recall: 0.9037 | Test F1: 0.9044

Epoch 3/20
------------------------------
Looked at 0 / 60000 samples
Looked at 12800 / 60000 samples
Looked at 25600 / 60000 samples
Looked at 38400 / 60000 samples
Looked at 51200 /

In [7]:
from src.models import CNN
from src.train import train_step, test_step

epochs = 20
history = {
    "train": [],
    "test": []
}

exp1 = CNN(dropout=0.2).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    params=exp1.parameters(),
    lr=0.0005
)

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    print("-"*30)

    train_results = train_step(
        model=exp1,
        dataloader=train_dataloader,
        loss_fn=loss_fn,
        optimizer=optimizer,
        metrics=train_metrics,
        device=device
    )

    test_results = test_step(
        model=exp1,
        dataloader=test_dataloader,
        loss_fn=loss_fn,
        metrics=test_metrics,
        device=device
    )

    history["train"].append(train_results)
    history["test"].append(test_results)

    print(
        f"Train Loss: {train_results['loss']:.4f} | "
        f"Train Acc: {train_results['accuracy']:.4f} | "
        f"Train Precision: {train_results['precision']:.4f} | "
        f"Train Recall: {train_results['recall']:.4f} | "
        f"Train F1: {train_results['f1']:.4f}"
    )

    print(
        f"Test Loss: {test_results['loss']:.4f} | "
        f"Test Acc: {test_results['accuracy']:.4f} | "
        f"Test Precision: {test_results['precision']:.4f} | "
        f"Test Recall: {test_results['recall']:.4f} | "
        f"Test F1: {test_results['f1']:.4f}"
    )

exp_results.append({
    "dropout": 0.2,
    "train_loss": history["train"][-1]["loss"],
    "test_loss": history["test"][-1]["loss"],
    "train_accuracy": history["train"][-1]["accuracy"],
    "test_accuracy": history["test"][-1]["accuracy"],
    "train_precision": history["train"][-1]["precision"],
    "test_precision": history["test"][-1]["precision"],
    "train_recall": history["train"][-1]["recall"],
    "test_recall": history["test"][-1]["recall"],
    "train_f1": history["train"][-1]["f1"],
    "test_f1": history["test"][-1]["f1"]
})


Epoch 1/20
------------------------------
Looked at 0 / 60000 samples
Looked at 12800 / 60000 samples
Looked at 25600 / 60000 samples
Looked at 38400 / 60000 samples
Looked at 51200 / 60000 samples
Train Loss: 0.4402 | Train Acc: 0.8414 | Train Precision: 0.8397 | Train Recall: 0.8414 | Train F1: 0.8402
Test Loss: 0.3339 | Test Acc: 0.8805 | Test Precision: 0.8839 | Test Recall: 0.8805 | Test F1: 0.8774

Epoch 2/20
------------------------------
Looked at 0 / 60000 samples
Looked at 12800 / 60000 samples
Looked at 25600 / 60000 samples
Looked at 38400 / 60000 samples
Looked at 51200 / 60000 samples
Train Loss: 0.2811 | Train Acc: 0.8967 | Train Precision: 0.8962 | Train Recall: 0.8966 | Train F1: 0.8963
Test Loss: 0.2672 | Test Acc: 0.9032 | Test Precision: 0.9029 | Test Recall: 0.9032 | Test F1: 0.9027

Epoch 3/20
------------------------------
Looked at 0 / 60000 samples
Looked at 12800 / 60000 samples
Looked at 25600 / 60000 samples
Looked at 38400 / 60000 samples
Looked at 51200 /

In [8]:
from src.models import CNN
from src.train import train_step, test_step

epochs = 20
history = {
    "train": [],
    "test": []
}

exp2 = CNN(dropout=0.4).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    params=exp2.parameters(),
    lr=0.0005
)

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    print("-"*30)

    train_results = train_step(
        model=exp2,
        dataloader=train_dataloader,
        loss_fn=loss_fn,
        optimizer=optimizer,
        metrics=train_metrics,
        device=device
    )

    test_results = test_step(
        model=exp2,
        dataloader=test_dataloader,
        loss_fn=loss_fn,
        metrics=test_metrics,
        device=device
    )

    history["train"].append(train_results)
    history["test"].append(test_results)

    print(
        f"Train Loss: {train_results['loss']:.4f} | "
        f"Train Acc: {train_results['accuracy']:.4f} | "
        f"Train Precision: {train_results['precision']:.4f} | "
        f"Train Recall: {train_results['recall']:.4f} | "
        f"Train F1: {train_results['f1']:.4f}"
    )

    print(
        f"Test Loss: {test_results['loss']:.4f} | "
        f"Test Acc: {test_results['accuracy']:.4f} | "
        f"Test Precision: {test_results['precision']:.4f} | "
        f"Test Recall: {test_results['recall']:.4f} | "
        f"Test F1: {test_results['f1']:.4f}"
    )

exp_results.append({
    "dropout": 0.4,
    "train_loss": history["train"][-1]["loss"],
    "test_loss": history["test"][-1]["loss"],
    "train_accuracy": history["train"][-1]["accuracy"],
    "test_accuracy": history["test"][-1]["accuracy"],
    "train_precision": history["train"][-1]["precision"],
    "test_precision": history["test"][-1]["precision"],
    "train_recall": history["train"][-1]["recall"],
    "test_recall": history["test"][-1]["recall"],
    "train_f1": history["train"][-1]["f1"],
    "test_f1": history["test"][-1]["f1"]
})


Epoch 1/20
------------------------------
Looked at 0 / 60000 samples
Looked at 12800 / 60000 samples
Looked at 25600 / 60000 samples
Looked at 38400 / 60000 samples
Looked at 51200 / 60000 samples
Train Loss: 0.4833 | Train Acc: 0.8266 | Train Precision: 0.8245 | Train Recall: 0.8266 | Train F1: 0.8249
Test Loss: 0.3162 | Test Acc: 0.8852 | Test Precision: 0.8851 | Test Recall: 0.8852 | Test F1: 0.8846

Epoch 2/20
------------------------------
Looked at 0 / 60000 samples
Looked at 12800 / 60000 samples
Looked at 25600 / 60000 samples
Looked at 38400 / 60000 samples
Looked at 51200 / 60000 samples
Train Loss: 0.3201 | Train Acc: 0.8847 | Train Precision: 0.8840 | Train Recall: 0.8847 | Train F1: 0.8842
Test Loss: 0.2782 | Test Acc: 0.9001 | Test Precision: 0.9005 | Test Recall: 0.9001 | Test F1: 0.8981

Epoch 3/20
------------------------------
Looked at 0 / 60000 samples
Looked at 12800 / 60000 samples
Looked at 25600 / 60000 samples
Looked at 38400 / 60000 samples
Looked at 51200 /

In [9]:
from src.models import CNN
from src.train import train_step, test_step

epochs = 20
history = {
    "train": [],
    "test": []
}

exp3 = CNN(dropout=0.5).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    params=exp3.parameters(),
    lr=0.0005
)

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    print("-"*30)

    train_results = train_step(
        model=exp3,
        dataloader=train_dataloader,
        loss_fn=loss_fn,
        optimizer=optimizer,
        metrics=train_metrics,
        device=device
    )

    test_results = test_step(
        model=exp3,
        dataloader=test_dataloader,
        loss_fn=loss_fn,
        metrics=test_metrics,
        device=device
    )

    history["train"].append(train_results)
    history["test"].append(test_results)

    print(
        f"Train Loss: {train_results['loss']:.4f} | "
        f"Train Acc: {train_results['accuracy']:.4f} | "
        f"Train Precision: {train_results['precision']:.4f} | "
        f"Train Recall: {train_results['recall']:.4f} | "
        f"Train F1: {train_results['f1']:.4f}"
    )

    print(
        f"Test Loss: {test_results['loss']:.4f} | "
        f"Test Acc: {test_results['accuracy']:.4f} | "
        f"Test Precision: {test_results['precision']:.4f} | "
        f"Test Recall: {test_results['recall']:.4f} | "
        f"Test F1: {test_results['f1']:.4f}"
    )

exp_results.append({
    "dropout": 0.5,
    "train_loss": history["train"][-1]["loss"],
    "test_loss": history["test"][-1]["loss"],
    "train_accuracy": history["train"][-1]["accuracy"],
    "test_accuracy": history["test"][-1]["accuracy"],
    "train_precision": history["train"][-1]["precision"],
    "test_precision": history["test"][-1]["precision"],
    "train_recall": history["train"][-1]["recall"],
    "test_recall": history["test"][-1]["recall"],
    "train_f1": history["train"][-1]["f1"],
    "test_f1": history["test"][-1]["f1"]
})


Epoch 1/20
------------------------------
Looked at 0 / 60000 samples
Looked at 12800 / 60000 samples
Looked at 25600 / 60000 samples
Looked at 38400 / 60000 samples
Looked at 51200 / 60000 samples
Train Loss: 0.4893 | Train Acc: 0.8272 | Train Precision: 0.8251 | Train Recall: 0.8272 | Train F1: 0.8258
Test Loss: 0.3130 | Test Acc: 0.8858 | Test Precision: 0.8877 | Test Recall: 0.8858 | Test F1: 0.8862

Epoch 2/20
------------------------------
Looked at 0 / 60000 samples
Looked at 12800 / 60000 samples
Looked at 25600 / 60000 samples
Looked at 38400 / 60000 samples
Looked at 51200 / 60000 samples
Train Loss: 0.3200 | Train Acc: 0.8864 | Train Precision: 0.8856 | Train Recall: 0.8864 | Train F1: 0.8858
Test Loss: 0.2667 | Test Acc: 0.9025 | Test Precision: 0.9029 | Test Recall: 0.9025 | Test F1: 0.9020

Epoch 3/20
------------------------------
Looked at 0 / 60000 samples
Looked at 12800 / 60000 samples
Looked at 25600 / 60000 samples
Looked at 38400 / 60000 samples
Looked at 51200 /

In [10]:
from src.models import CNN
from src.train import train_step, test_step

epochs = 20
history = {
    "train": [],
    "test": []
}

exp4 = CNN(dropout=0.6).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    params=exp4.parameters(),
    lr=0.0005
)

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    print("-"*30)

    train_results = train_step(
        model=exp4,
        dataloader=train_dataloader,
        loss_fn=loss_fn,
        optimizer=optimizer,
        metrics=train_metrics,
        device=device
    )

    test_results = test_step(
        model=exp4,
        dataloader=test_dataloader,
        loss_fn=loss_fn,
        metrics=test_metrics,
        device=device
    )

    history["train"].append(train_results)
    history["test"].append(test_results)

    print(
        f"Train Loss: {train_results['loss']:.4f} | "
        f"Train Acc: {train_results['accuracy']:.4f} | "
        f"Train Precision: {train_results['precision']:.4f} | "
        f"Train Recall: {train_results['recall']:.4f} | "
        f"Train F1: {train_results['f1']:.4f}"
    )

    print(
        f"Test Loss: {test_results['loss']:.4f} | "
        f"Test Acc: {test_results['accuracy']:.4f} | "
        f"Test Precision: {test_results['precision']:.4f} | "
        f"Test Recall: {test_results['recall']:.4f} | "
        f"Test F1: {test_results['f1']:.4f}"
    )

exp_results.append({
    "dropout": 0.6,
    "train_loss": history["train"][-1]["loss"],
    "test_loss": history["test"][-1]["loss"],
    "train_accuracy": history["train"][-1]["accuracy"],
    "test_accuracy": history["test"][-1]["accuracy"],
    "train_precision": history["train"][-1]["precision"],
    "test_precision": history["test"][-1]["precision"],
    "train_recall": history["train"][-1]["recall"],
    "test_recall": history["test"][-1]["recall"],
    "train_f1": history["train"][-1]["f1"],
    "test_f1": history["test"][-1]["f1"]
})


Epoch 1/20
------------------------------
Looked at 0 / 60000 samples
Looked at 12800 / 60000 samples
Looked at 25600 / 60000 samples
Looked at 38400 / 60000 samples
Looked at 51200 / 60000 samples
Train Loss: 0.5905 | Train Acc: 0.7885 | Train Precision: 0.7855 | Train Recall: 0.7885 | Train F1: 0.7862
Test Loss: 0.3476 | Test Acc: 0.8721 | Test Precision: 0.8733 | Test Recall: 0.8721 | Test F1: 0.8702

Epoch 2/20
------------------------------
Looked at 0 / 60000 samples
Looked at 12800 / 60000 samples
Looked at 25600 / 60000 samples
Looked at 38400 / 60000 samples
Looked at 51200 / 60000 samples
Train Loss: 0.3933 | Train Acc: 0.8602 | Train Precision: 0.8591 | Train Recall: 0.8602 | Train F1: 0.8592
Test Loss: 0.2999 | Test Acc: 0.8905 | Test Precision: 0.8907 | Test Recall: 0.8905 | Test F1: 0.8885

Epoch 3/20
------------------------------
Looked at 0 / 60000 samples
Looked at 12800 / 60000 samples
Looked at 25600 / 60000 samples
Looked at 38400 / 60000 samples
Looked at 51200 /

In [11]:
from src.models import CNN
from src.train import train_step, test_step

epochs = 20
history = {
    "train": [],
    "test": []
}

exp5 = CNN(dropout=0.8).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    params=exp5.parameters(),
    lr=0.0005
)

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    print("-"*30)

    train_results = train_step(
        model=exp5,
        dataloader=train_dataloader,
        loss_fn=loss_fn,
        optimizer=optimizer,
        metrics=train_metrics,
        device=device
    )

    test_results = test_step(
        model=exp5,
        dataloader=test_dataloader,
        loss_fn=loss_fn,
        metrics=test_metrics,
        device=device
    )

    history["train"].append(train_results)
    history["test"].append(test_results)

    print(
        f"Train Loss: {train_results['loss']:.4f} | "
        f"Train Acc: {train_results['accuracy']:.4f} | "
        f"Train Precision: {train_results['precision']:.4f} | "
        f"Train Recall: {train_results['recall']:.4f} | "
        f"Train F1: {train_results['f1']:.4f}"
    )

    print(
        f"Test Loss: {test_results['loss']:.4f} | "
        f"Test Acc: {test_results['accuracy']:.4f} | "
        f"Test Precision: {test_results['precision']:.4f} | "
        f"Test Recall: {test_results['recall']:.4f} | "
        f"Test F1: {test_results['f1']:.4f}"
    )

exp_results.append({
    "dropout": 0.8,
    "train_loss": history["train"][-1]["loss"],
    "test_loss": history["test"][-1]["loss"],
    "train_accuracy": history["train"][-1]["accuracy"],
    "test_accuracy": history["test"][-1]["accuracy"],
    "train_precision": history["train"][-1]["precision"],
    "test_precision": history["test"][-1]["precision"],
    "train_recall": history["train"][-1]["recall"],
    "test_recall": history["test"][-1]["recall"],
    "train_f1": history["train"][-1]["f1"],
    "test_f1": history["test"][-1]["f1"]
})


Epoch 1/20
------------------------------
Looked at 0 / 60000 samples
Looked at 12800 / 60000 samples
Looked at 25600 / 60000 samples
Looked at 38400 / 60000 samples
Looked at 51200 / 60000 samples
Train Loss: 0.8356 | Train Acc: 0.6846 | Train Precision: 0.6773 | Train Recall: 0.6846 | Train F1: 0.6799
Test Loss: 0.3982 | Test Acc: 0.8593 | Test Precision: 0.8565 | Test Recall: 0.8593 | Test F1: 0.8556

Epoch 2/20
------------------------------
Looked at 0 / 60000 samples
Looked at 12800 / 60000 samples
Looked at 25600 / 60000 samples
Looked at 38400 / 60000 samples
Looked at 51200 / 60000 samples
Train Loss: 0.6124 | Train Acc: 0.7622 | Train Precision: 0.7591 | Train Recall: 0.7622 | Train F1: 0.7601
Test Loss: 0.3453 | Test Acc: 0.8776 | Test Precision: 0.8762 | Test Recall: 0.8776 | Test F1: 0.8756

Epoch 3/20
------------------------------
Looked at 0 / 60000 samples
Looked at 12800 / 60000 samples
Looked at 25600 / 60000 samples
Looked at 38400 / 60000 samples
Looked at 51200 /

In [21]:
df = pd.DataFrame(exp_results)
df = df.sort_values(
    by=["dropout"],
    ascending=[True]
)
df

,dropout,train_loss,test_loss,train_accuracy,test_accuracy,train_precision,test_precision,train_recall,test_recall,train_f1,test_f1
5,0.0,0.008142,0.541725,0.997067,0.9227,0.997066,0.923460,0.997067,0.9227,0.997066,0.923008
0,0.2,0.028744,0.413835,0.989233,0.9270,0.989231,0.926999,0.989233,0.9270,0.989231,0.926772
1,0.4,0.054595,0.321659,0.978517,0.9275,0.978495,0.927471,0.978517,0.9275,0.978491,0.927437
2,0.5,0.057100,0.306810,0.977733,0.9309,0.977722,0.930795,0.977733,0.9309,0.977725,0.930805
3,0.6,0.114456,0.267079,0.954567,0.9276,0.954624,0.928588,0.954567,0.9276,0.954583,0.927804
4,0.8,0.277591,0.228771,0.891300,0.9266,0.892273,0.926398,0.891300,0.9266,0.891295,0.926230


In [22]:
new_df = df[['dropout', 'train_loss', 'test_loss', 'train_accuracy', 'test_accuracy']]

new_df

,dropout,train_loss,test_loss,train_accuracy,test_accuracy
5,0.0,0.008142,0.541725,0.997067,0.9227
0,0.2,0.028744,0.413835,0.989233,0.9270
1,0.4,0.054595,0.321659,0.978517,0.9275
2,0.5,0.057100,0.306810,0.977733,0.9309
3,0.6,0.114456,0.267079,0.954567,0.9276
4,0.8,0.277591,0.228771,0.891300,0.9266
